# Clean CV Data

- Parse Filenames. These rules should automatically parse most naming conventions.

  - `Separator` refers to `-`, `_`, space, newline, beginning of string, or end of string.

  - `wafer_num` : Within the username, match the strings `wafer`, `waf`, or `w` followed  
  by zero or one separator before the wafer number. Preceded and followed by separators.  
  `-1` if no match.
  - `position`  : Within the username excluding previous matches, match the strings `c`,  
  `cen`, `center`, `m`, `mid`, `middle`, `b`, `bot`, `bottom`, `t`, `top`, `l`, `left`,  
  `r`, `right`. Preceded and followed by separators. `"NOMATCH"` if no match.  
  - `cap_diam`  : Within the username excluding previous matches, match any number in  
  [100, 1000] step 100 or [0.1, 1.0] step 0.1, optionally followed by `um`, `u` or `mm`, `m`  
  respectively. Preceded and followed by separators. `-1` if no match.  
  - `cap_ind`   : Within the username excluding previous matches, match the first integer,  
  optionally preceeded by `c`, `cap`, `capacitor`, `p`, `pos`, `position`. Preceeded and  
  followed by separators.
    - You might be indexing within rows, sections, or wafers.  
  - `flags`     : Optional. Within the username excluding previous matches, match any  
  separator, set of lowercase alphabetical characters, and another separator.

- Export Parsed Information.

  - `Selected_Measurements.csv`
    - Indicies: Run Numbers
    - Columns: `username`, `wafer_num`, `pos`, `cap_diam` (um), `cap_ind`, `flags`

- Export Aggregated CSVs.

  - `Cp_by_Measurement.csv`
    - Indicies: Run Numbers
    - Columns: The same as `Selected_Measurements` plus each point of the voltage sweep.
    - Data: Same as `Selected_Measurements` plus the <u>**capacitance**</u> at each voltage.

  - `Gp_by_Measurement.csv`
    - Indicies: Run Numbers
    - Columns: The same as `Selected_Measurements` plus each point of the voltage sweep.
    - Data: Same as `Selected_Measurements` plus the <u>**conductance**</u> at each voltage.

## Preparation

### Import Modules

In [11]:
import os
import re

import numpy as np
import pandas as pd

### Parameters

In [12]:
RAW_DATA_PATH = "Raw"

## Clean Data

### Get Measurement Indicies

In [13]:
run_indicies    = sorted([int(dirName.lstrip("Run")) for dirName in os.listdir(RAW_DATA_PATH) if "Run" in dirName])
run_dir_names   = [f"Run{ri}" for ri in run_indicies]

### Pick Out Measurements with Names

In [30]:
# Should work with most naming conventions
# Formatted: Capturing group, regex pattern
username_re = 1, re.compile(r"<Run.*username=\"(.*)\".*>")
wafer_re    = 3, re.compile(r"(^|[-_\s])(wafer|waf|w)[-_\s]?([0-9]+)([-_\s]|\n|$)", re.IGNORECASE)
pos_re      = 2, re.compile(r"(^|[-_\s])(b(ottom|ot)?|l(eft)?|t(op)?|r(ight)?|c(enter|en)?|m(iddle|id)?)([-_\s]|\n|$)", re.IGNORECASE)
cap_diam_re = 2, re.compile(r"(^|[-_\s])((1\.0+|0?\.[1-9])|(1000|[1-9]00))([a-z]*)([-_\s]|\n|$)?", re.IGNORECASE)
cap_ind_re  = 6, re.compile(r"(^|[-_\s])(c(apacitor|ap)?|s(ample|amp)?|p(osition|os)?)?([0-9]+)([a-z]*)([-_\s]|\n|$)", re.IGNORECASE)
flags_re    = 1, re.compile(r"[-_\s]([a-z]+)(\n|$)")

In [31]:
useful_measurements = {
    "idx"       : [],
    "username"  : [],
    "wafer_num" : [],
    "pos"       : [],
    "cap_diam"  : [],
    "cap_ind"   : [],
    "flags"     : [],
}

for ri, rdn in zip(run_indicies, run_dir_names):
    run_xml_name = "run.xml"
    username     = ""
    with open(os.sep.join([RAW_DATA_PATH, rdn, run_xml_name])) as run_xml:
        run_xml.readline()
        username_match = re.match(username_re[1], run_xml.readline())
        username = username_match.group(username_re[0]) if username_match is not None else ""

    if not len(username):
        continue
    
    username_og = username

    username        += "_d" if "dep" in username or "deposition" in username else ""

    wafer_num_match = re.search(wafer_re[1], username)
    wafer_num       = int(wafer_num_match.group(wafer_re[0])) if wafer_num_match is not None else -1
    username        = re.sub(wafer_re[1], "_WAFERNUM_", username)

    if wafer_num_match is None:
        continue

    pos_match       = re.search(pos_re[1], username)
    pos             = pos_match.group(pos_re[0]).upper() if pos_match is not None else ""
    pos             = \
        "NOMATCH"   if len(pos) == 0  else \
        "CENTER"    if pos[0] in "CM" else \
        "BOTTOM"    if pos[0] == "B"  else \
        "TOP"       if pos[0] == "T"  else \
        "LEFT"      if pos[0] == "L"  else \
        "RIGHT"     if pos[0] == "R"  else \
        "UNKNOWN"
    username        = re.sub(pos_re[1], "_POSITION_", username)

    cap_diam_match  = re.search(cap_diam_re[1], username)
    cap_diam        = cap_diam_match.group(cap_diam_re[0]) if cap_diam_match is not None else -1
    cap_diam        = float(cap_diam) * (1e3 if "." in cap_diam else 1) if type(cap_diam) is str else -1
    username        = re.sub(cap_diam_re[1], "_CAP-SIZE_", username)

    cap_ind_match   = re.search(cap_ind_re[1], username)
    cap_ind         = int(cap_ind_match.group(cap_ind_re[0])) if cap_ind_match is not None else -1
    username        = re.sub(cap_ind_re[1], "_CAP-IND_", username)

    flags_match     = re.search(flags_re[1], username)
    flags           = flags_match.group(flags_re[0]) if flags_match is not None else ""
    username        = re.sub(flags_re[1], "_FLAGS_", username)

    username        = username.strip("_")
    print(f"Index: {ri:4d} | Name: {username_og:25s} -> WAFER = {wafer_num:2d}, POSITION = {pos:8s}, CAP_DIAM = {cap_diam:7.1f} um, CAP_IND = {cap_ind:2d} | Detected Naming: {username:40s}")

    useful_measurements[ "idx"       ].append(ri)
    useful_measurements[ "username"  ].append(username_og)
    useful_measurements[ "wafer_num" ].append(wafer_num)
    useful_measurements[ "pos"       ].append(pos)
    useful_measurements[ "cap_diam"  ].append(cap_diam)
    useful_measurements[ "cap_ind"   ].append(cap_ind)
    useful_measurements[ "flags"     ].append(flags)

Index:   12 | Name: Wafer5_mid_400d           -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   400.0 um, CAP_IND = -1 | Detected Naming: WAFERNUM_POSITION_CAP-SIZE              
Index:   13 | Name: Wafer5_mid_400d_p2        -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   400.0 um, CAP_IND =  2 | Detected Naming: WAFERNUM_POSITION_CAP-SIZE_CAP-IND      
Index:   15 | Name: Wafer5_mid_400d_p3        -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   400.0 um, CAP_IND =  3 | Detected Naming: WAFERNUM_POSITION_CAP-SIZE_CAP-IND      
Index:   23 | Name: Wafer5_mid_300d_p1        -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   300.0 um, CAP_IND =  1 | Detected Naming: WAFERNUM_POSITION_CAP-SIZE_CAP-IND      
Index:   27 | Name: Wafer5_mid_300d_p2        -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   300.0 um, CAP_IND =  2 | Detected Naming: WAFERNUM_POSITION_CAP-SIZE_CAP-IND      
Index:   28 | Name: Wafer5_mid_300d_p3_d      -> WAFER =  5, POSITION = CENTER  , CAP_DIAM =   300.0 um, CAP_I

### Build Dataframe for Desired Measurements

In [6]:
measurements_info_df = pd.DataFrame(useful_measurements, index=useful_measurements["idx"]).drop(columns="idx")

Clean Data How You Like

In [7]:
measurements_info_df.loc[measurements_info_df.loc[:, "cap_ind"] == -1, "cap_ind"] = 1
measurements_info_df.sort_values(["cap_diam", "wafer_num", "pos", "cap_ind"], inplace=True)

Save Dataframe

In [8]:
measurements_info_df.to_csv("Selected_Measurements.csv")

## Aggregate CV Curves
- Horizontal Axis: Voltages
- Vertical Axis: Measurement Run Number
- Inside: Cp or Gp data.

In [38]:
Gp_df = measurements_info_df.copy()
Cp_df = measurements_info_df.copy()

for ri in useful_measurements["idx"]:
    rdn             = f"Run{ri}"
    excel_fileName  = [ fn for fn in os.listdir(os.sep.join([RAW_DATA_PATH, rdn])) if ".xls" in fn ][0]
    data_this_cap   = pd.read_excel(os.sep.join([RAW_DATA_PATH, rdn, excel_fileName]))
    voltages        = [ f"{x:.2f}" for x in data_this_cap["DCV_AB"].values ]
    Gp_df.loc[ri, voltages] = [f"{d:.6e}" for d in data_this_cap["Gp_AB"].values]
    Cp_df.loc[ri, voltages] = [f"{d:.6e}" for d in data_this_cap["Cp_AB"].values]

WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (26770) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero

In [39]:
Gp_df.to_csv("Gp_by_Measurement.csv", index=True)
Cp_df.to_csv("Cp_by_Measurement.csv", index=True)